# 11 Export: Charts 8–11: who the game misjudged

**Owner:** Sarthak

Writes one upload-ready CSV per chart to `outputs/charts/`, its spec entry to `outputs/charts/chart_specs.md`, and the numbers its title uses to `outputs/findings.json` (`export_chart_NN`). Chart tables are built by `src/exports.py` from `data/clean/analysis.parquet` and the notebook 05 findings. Titles are assembled from those numbers, never typed by hand.

Run notebook 05 first. Featured five: `findings.json → featured_five`.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import config, exports as ex, findings as fd

pd.set_option("display.max_columns", None); pd.set_option("display.width", 220)
NB = "11_export_act3"
P = config.PALETTE
ctx = ex.load_ctx()
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append({"Check": name, "Result": "PASS" if passed else "FAIL", "Detail": detail})
    print(("PASS" if passed else "FAIL"), "-", name, ("| " + detail if detail else ""))

def publish(n, short, df, facts, spec):
    check(f"Chart {n:02d}: under {config.CHART_MAX_ROWS} rows", len(df) <= config.CHART_MAX_ROWS, f"{len(df)} rows")
    ex.save_csv(n, short, df)
    fd.update(f"export_chart_{n:02d}", facts, NB)
    ex.write_spec(n, spec)
    print(f"wrote outputs/charts/{n:02d}_{short}.csv ({len(df)} rows) and its spec")
    display(df.head(8))
print("Featured five:", list(ctx.featured_names.values()))

Featured five: ['R. Sterling', 'R. Živković', 'H. Kane', 'M. Depay', 'T. Werner']


## Chart 08: nation_map

In [2]:
nation_iso = pd.read_parquet(config.CLEAN / "nation_iso.parquet")
df, f = ex.chart_08(ctx, nation_iso)
check("Chart 08: one row per ISO3", df["ISO3"].is_unique)
ms, mb = f["most_short_100plus"], f["most_beat_100plus"]
SHORT = {"Republic of Ireland": "Ireland", "United Kingdom": "the UK", "United States": "the US", "Korea Republic": "South Korea"}
f["top3_beat_100plus"] = [SHORT.get(x, x) for x in f["top3_beat_100plus"]]
f["top3_fell_short_100plus"] = [SHORT.get(x, x) for x in f["top3_fell_short_100plus"]]
spec = ex.spec_entry(8, "nation_map", "Datawrapper choropleth map (World)",
    f"Prospects from {', '.join(f['top3_beat_100plus'][:-1])} and {f['top3_beat_100plus'][-1]} beat their promise; those from {', '.join(f['top3_fell_short_100plus'][:-1])} and {f['top3_fell_short_100plus'][-1]} fell short",
    "Shortfall vs similar prospects: how many rating points a nation's entrants peaked below (+) or above (−) players given the same potential. Nations with 15+ entrants (named in the title: 100+ and 95% CI excluding 0); others blank",
    "match key = ISO3 (ISO 3166-1 alpha-3); value = Shortfall vs similar prospects; tooltip = all columns",
    f"diverging scale centred on 0: negative (beat) {P['beat']}, positive (fell short) {P['fell_short']}, 0 white; no data {P['context']} at 40%",
    "<b>{{ Nation }}</b> ({{ Players }} players)<br>Shortfall vs similar prospects: {{ Shortfall vs similar prospects }} points<br>95% CI {{ CI low }} to {{ CI high }}<br>{{ Verdict }}<br>Reached potential: {{ Share reached potential (%) }}%",
    "Caveat for the text: nationality overlaps with where players play (many British and Irish entrants were in English League One/Two, where all prospects fell short), so say 'prospects from', not 'EA is biased against'. England, Scotland, Wales and Northern Ireland are pooled as 'United Kingdom' (GBR) because the world map has one shape for the UK (DECISIONS.md). Use 5 classes, symmetric around 0 (e.g. −3, −1, 1, 3). Only nations whose 95% CI excludes 0 are distinguishable: mention that in the notes and show the Verdict in the tooltip. Kosovo (XKX) has fewer than 15 entrants, so it's not in the file.")
publish(8, "nation_map", df, f, spec)

PASS - Chart 08: one row per ISO3 
PASS - Chart 08: under 3000 rows | 53 rows
wrote outputs/charts/08_nation_map.csv (53 rows) and its spec


,ISO3,Nation,Shortfall vs similar prospects,CI low,CI high,Verdict,Share reached potential (%),Players
25,MLI,Mali,-3.6,-6.1,-1.2,Beat similar prospects,52,21
51,URY,Uruguay,-3.3,-4.9,-1.7,Beat similar prospects,35,49
13,CIV,Côte d'Ivoire,-3.0,-4.7,-1.5,Beat similar prospects,42,50
5,BIH,Bosnia and Herzegovina,-2.9,-4.7,-0.9,Beat similar prospects,52,23
0,ALB,Albania,-2.2,-5.5,1.2,In line with similar prospects,47,17
42,SVK,Slovakia,-2.2,-4.5,-0.0,Beat similar prospects,38,26
6,BRA,Brazil,-2.2,-3.0,-1.4,Beat similar prospects,30,176
44,ESP,Spain,-2.0,-2.5,-1.5,Beat similar prospects,36,500


## Chart 09: age_at_entry

In [3]:
df, f = ex.chart_09(ctx)
spec = ex.spec_entry(9, "age_at_entry", "Flourish column chart, two stacked panels (no second axis)",
    f"17-year-olds were promised {f['headroom_17']} points of growth, nearly twice a 21-year-old's, and fell {f['shortfall_17_minus_21']:.1f} points further short",
    f"Columns: shortfall vs similar prospects by age at entry (rating points; + = fell short of players given the same potential). Line: share who made it in real life, {f['made_it_17']}% at 17 vs {f['made_it_21']}% at 21: the extra promise wasn't earned",
    "Panel 1: x = Age at entry, columns = Shortfall vs similar prospects, error bars = CI low / CI high. Panel 2 (small, below, same x): columns = Made it in real life (%). Never put the two measures on one plot with two y-axes",
    f"columns positive {P['fell_short']}, negative {P['beat']}; made-it line {P['reality']}; headroom shown in the tooltip",
    "<b>Age {{ Age at entry }}</b> ({{ Players }} players)<br>Headroom given: {{ Headroom given (potential − overall) }} points<br>Shortfall vs similar prospects: {{ Shortfall vs similar prospects }} (95% CI {{ CI low }} to {{ CI high }})<br>Made it in real life: {{ Made it in real life (%) }}%",
    f"Build panel 2 as a second Flourish column chart (or a Flourish 'small multiples' grid) placed directly under panel 1. Flourish 'Column chart' with error bars (Data > Error bars: CI low / CI high). If error bars are awkward, use a Datawrapper range plot instead and say so. The age-16 bar is based on only 72 players (wide CI). The difference between ages ≤17 and 21 is {f['shortfall_17_minus_21']} points (95% CI {f['ci'][0]}–{f['ci'][1]}).")
publish(9, "age_at_entry", df, f, spec)

PASS - Chart 09: under 3000 rows | 6 rows
wrote outputs/charts/09_age_at_entry.csv (6 rows) and its spec


,Age at entry,Headroom given (potential − overall),Shortfall vs similar prospects,CI low,CI high,Made it in real life (%),Players
0,16,14.8,2.1,0.4,3.7,14,72
1,17,15.1,1.8,1.3,2.3,12,862
2,18,13.5,1.3,1.0,1.6,12,1790
3,19,11.8,0.5,0.3,0.8,11,2088
4,20,9.8,-0.9,-1.2,-0.7,10,2041
5,21,8.1,-1.7,-1.9,-1.5,11,1908


## Chart 10: club_badge

In [4]:
df, f = ex.chart_10(ctx)
p_ = f["badge_premium_per_100_elo"]
spec = ex.spec_entry(10, "club_badge", "Flourish scatter",
    f"A big badge bought almost no extra promise ({f['premium_p10_to_p90']} potential points from a weak to an elite club), and big-club prospects lived up to it",
    "One dot per club with 5+ entrants rated by ClubElo at entry. x = mean club Elo when the player entered the game; y = shortfall vs similar prospects (+ = fell short)",
    "x = Mean club Elo at entry; y = Shortfall vs similar prospects; size = Players; label = Club (top clubs only); tooltip adds Extra potential and League",
    f"dots {P['context']}, clubs with |shortfall| ≥ 3 in {P['fell_short']} (positive) or {P['beat']} (negative); y = 0 reference line in {P['text']}",
    "<b>{{ Club }}</b> ({{ League }}, {{ Country }})<br>{{ Players }} entrants · mean Elo {{ Mean club Elo at entry }}<br>Shortfall vs similar prospects: {{ Shortfall vs similar prospects }}<br>Extra potential vs same rating and age: {{ Extra potential vs same rating and age }}",
    f"Weak vs elite club = 10th vs 90th percentile entry Elo ({f['elo_p10']} vs {f['elo_p90']}). Badge premium: {p_['value']} potential points per +100 Elo for the same overall and age (95% CI {p_['ci_low']}–{p_['ci_high']}). Club-level Spearman(Elo, shortfall) = {f['spearman_club_elo_vs_shortfall']}. Known gaps (no Elo): Swiss, Irish, Ukrainian and Czech clubs, English League One/Two and reserve sides (DECISIONS.md). Source line credits the ClubElo mirror and ClubElo.")
publish(10, "club_badge", df, f, spec)

PASS - Chart 10: under 3000 rows | 397 rows
wrote outputs/charts/10_club_badge.csv (397 rows) and its spec


,Club,League,Country,Mean club Elo at entry,Shortfall vs similar prospects,Extra potential vs same rating and age,Players
281,Real Madrid,La Liga,Spain,2058,-5.0,0.2,25
111,FC Barcelona,La Liga,Spain,2043,-3.8,-1.7,8
112,FC Bayern München,Bundesliga,Germany,2010,-0.2,2.1,15
31,Atlético Madrid,La Liga,Spain,1976,-1.1,0.1,17
213,Manchester City,Premier League,England,1916,-0.2,4.3,25
175,Juventus,Serie A,Italy,1895,0.2,1.4,14
75,Chelsea,Premier League,England,1893,-3.6,2.0,23
261,Paris Saint Germain,Ligue 1,France,1866,-5.6,-0.8,19


## Chart 11: position_gap

In [5]:
df, f = ex.chart_11(ctx)
W = {"GK": "Goalkeepers", "ST": "Strikers", "CF": "Centre-forwards", "CDM": "Defensive midfielders", "LW": "Left wingers", "RW": "Right wingers", "CB": "Centre-backs", "LB": "Left-backs", "RB": "Right-backs", "CM": "Central midfielders", "CAM": "Attacking midfielders", "LM": "Left midfielders", "RM": "Right midfielders"}
spec = ex.spec_entry(11, "position_gap", "Flourish bar chart (horizontal, with error bars)",
    f"{W[f['worst']['Primary position']]} fell furthest short of their promise; {W[f['best']['Primary position']].lower()} beat theirs",
    "Shortfall vs similar prospects by primary position at entry (rating points; + = peaked below players given the same potential). Positions with 30+ entrants",
    "y = Primary position (sorted); x = Shortfall vs similar prospects; error bars = CI low / CI high; colour = sign",
    f"positive {P['fell_short']}, negative {P['beat']}; positions whose CI crosses 0 in {P['context']}",
    "<b>{{ Primary position }}</b> ({{ Players }} players)<br>Shortfall vs similar prospects: {{ Shortfall vs similar prospects }} (95% CI {{ CI low }} to {{ CI high }})<br>Mean potential {{ Mean potential at entry }} → mean peak {{ Mean peak later rating }}",
    f"Positions clearly short of their promise: {', '.join(f['fell_short'])}; clearly beating it: {', '.join(f['beat'])}. If Flourish error bars are awkward, use a Datawrapper dot plot with a range and swap tools with chart 12.")
publish(11, "position_gap", df, f, spec)

PASS - Chart 11: under 3000 rows | 13 rows
wrote outputs/charts/11_position_gap.csv (13 rows) and its spec


,Primary position,Position group,Shortfall vs similar prospects,CI low,CI high,Mean potential at entry,Mean peak later rating,Players
5,GK,GK,1.1,0.6,1.5,68,65,917
12,ST,FWD,0.9,0.5,1.2,70,66,1425
3,CF,FWD,0.6,-0.9,2.0,72,67,70
4,CM,MID,0.1,-0.3,0.5,70,66,1229
11,RW,FWD,0.1,-0.6,0.8,73,68,283
0,CAM,MID,-0.2,-0.6,0.3,72,68,715
10,RM,MID,-0.3,-0.9,0.2,71,68,437
7,LM,MID,-0.4,-0.9,0.2,71,68,452


## Summary

In [6]:
checks = pd.DataFrame(CHECKS); display(checks)
assert (checks["Result"] == "PASS").all()

,Check,Result,Detail
0,Chart 08: one row per ISO3,PASS,
1,Chart 08: under 3000 rows,PASS,53 rows
2,Chart 09: under 3000 rows,PASS,6 rows
3,Chart 10: under 3000 rows,PASS,397 rows
4,Chart 11: under 3000 rows,PASS,13 rows
